# Снижение размерности и поиск фенотипов

Разведочный неконтролируемый анализ: PCA, t-SNE, UMAP и кластеризация по очищенным данным. Целевая переменная (рецидив) используется только для раскраски точек, не для обучения. Логика в `src/dimreduce.py`.

Матрица: количественные показатели с долей пропусков не выше 40%, скошенные логарифмированы, пропуски заполнены медианой (грубо, только для визуализации), затем стандартизация. После этапа 5 (честная импутация) анализ повторим.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

from IPython.display import Image, display
from src import io, columns, dimreduce, config

dimreduce.apply_style()
df = io.load_processed()
target = columns.classify(df)['target']
target_series = df[target]
X, used = dimreduce.build_matrix(df)
print('Признаков в матрице:', len(used))
used

## Матрицы корреляций количественных показателей

Корреляции считаем на сырых данных, без заполнения пропусков, попарно: каждый коэффициент оценивается по наблюдениям, где оба показателя присутствуют (минимум 20 совпадающих наблюдений). Так корреляции не зависят от выбранной импутации. Строим две матрицы: Спирмена (ранговая, устойчива к скошенности и выбросам, основная для этих данных) и Пирсона (линейная). Высокие по модулю значения указывают на мультиколлинеарность, которую учитываем при отборе признаков (например возраст, длительность СД и возраст манифестации связаны почти тождественно).

In [ ]:
corr = dimreduce.correlation_heatmaps(df)
display(Image(str(config.FIGURES_DIR / 'corr_spearman.png')))
display(Image(str(config.FIGURES_DIR / 'corr_pearson.png')))

## PCA

Сколько компонент нужно, чтобы объяснить дисперсию, и как пациенты ложатся в плоскость первых двух компонент.

In [ ]:
pca_res = dimreduce.run_pca(X, target_series)
print('Доля дисперсии по компонентам, %:',
      [round(v * 100, 1) for v in pca_res['explained'][:8]])
display(Image(str(config.FIGURES_DIR / 'pca_scree.png')))
display(Image(str(config.FIGURES_DIR / 'pca_scatter.png')))

## t-SNE и UMAP

Нелинейные проекции для оценки структуры. Важно: абсолютные расстояния и размеры скоплений тут не интерпретируются, это только визуализация.

In [ ]:
dimreduce.run_tsne(X, target_series)
dimreduce.run_umap(X, target_series)
display(Image(str(config.FIGURES_DIR / 'tsne_scatter.png')))
display(Image(str(config.FIGURES_DIR / 'umap_scatter.png')))

## Кластеризация

KMeans с подбором числа кластеров по силуэту. Сверяем кластеры с фактом рецидива: если кластеры не разделяют классы, значит в неконтролируемом пространстве рецидив не выражен и нужны управляемые модели.

In [ ]:
clust = dimreduce.run_clustering(X, pca_res['scores'], target_series)
print('Силуэт по k:', {k: round(v, 3) for k, v in clust['silhouette'].items()})
print('Лучшее k:', clust['best_k'])
display(Image(str(config.FIGURES_DIR / 'clusters_pca.png')))
clust['crosstab']